In [39]:
import numpy as np
import xarray as xr
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, interp2d
from matplotlib import cm,colors
import pickle
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import pandas as pd


In [40]:
melange_area_path = '../../data/melange_area_geoms/'
mel_list = sorted( [file for file in os.listdir(melange_area_path) if file.endswith('gpkg') ] )
date_list = [gpd.pd.to_datetime(file[8:18]) for file in mel_list]
gdf_list = [gpd.read_file(f'{melange_area_path}{file}') for file in mel_list]

In [41]:
mel_list

['helhiem_2016-04-24_area.gpkg',
 'helhiem_2018-05-04_area.gpkg',
 'helhiem_2020-09-03_area.gpkg',
 'helhiem_2023-07-27_area.gpkg',
 'helhiem_2024-08-22_area.gpkg']

In [42]:
date_list

[Timestamp('2016-04-24 00:00:00'),
 Timestamp('2018-05-04 00:00:00'),
 Timestamp('2020-09-03 00:00:00'),
 Timestamp('2023-07-27 00:00:00'),
 Timestamp('2024-08-22 00:00:00')]

In [43]:
gdf_list[0].area[0]

np.float64(132528531.77252682)

In [44]:
area_dict = {date:gdf.area[0] for date, gdf in zip(date_list, gdf_list)}
area_dict

{Timestamp('2016-04-24 00:00:00'): np.float64(132528531.77252682),
 Timestamp('2018-05-04 00:00:00'): np.float64(142934146.47607896),
 Timestamp('2020-09-03 00:00:00'): np.float64(141325864.68025035),
 Timestamp('2023-07-27 00:00:00'): np.float64(142934146.47607896),
 Timestamp('2024-08-22 00:00:00'): np.float64(134061538.63418496)}

In [45]:
temp_dict = {'min':4.0,
            'avg': 5.4,
            'max': 6.3} #from CTD AW average TF

urel_dict = {'min':0.07,
            'avg': 0.13,
            'max': 0.20} #model vel runs

In [48]:
run_type = 'min'

dt = 50
psw = 1024 #kg m3
csw = 3974 #J kg-1 C-1
day2sec = 86400
depth = 450
temp = temp_dict[run_type]
coeff_1_path = f'../../data/iceberg_model_output/helheim/{run_type}/'


coeff_1_list = sorted([nc for nc in os.listdir(coeff_1_path) if nc.endswith('nc')])

def Qaw_calc(area, dt = dt, psw = psw, csw = csw, depth = depth, temp = temp):
    
    vol = area * depth
    
    Qaw = psw * csw * ( (vol * temp) / (dt * day2sec) )

    return Qaw

dQ_dt_HEL_CTD_avg_dict = {date:Qaw_calc(vol) for date, vol in area_dict.items()}

cols = ['coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'melt_rate_mday', 'percentage']

In [49]:
dQ_dt_HEL_CTD_avg_dict

{Timestamp('2016-04-24 00:00:00'): np.float64(224711844379.3159),
 Timestamp('2018-05-04 00:00:00'): np.float64(242355327187.6001),
 Timestamp('2020-09-03 00:00:00'): np.float64(239628367462.10767),
 Timestamp('2023-07-27 00:00:00'): np.float64(242355327187.6001),
 Timestamp('2024-08-22 00:00:00'): np.float64(227311169933.76044)}

In [50]:
# dQ_dt_HEL_CTD_avg/1e9

In [51]:
cols = ['date', 'coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'percentage']
coef_1_dict = {}

series_list1 = []
for i,nc in enumerate(coeff_1_list):
    
    Qib = xr.open_dataset(f'{coeff_1_path}{nc}')
    Qib_val = Qib.Qib.data
    date = nc[:10]
    date = pd.to_datetime(date)
    TF = nc.split('_')[6]
    urel_val = nc.split('_')[-1].split('.')[1]
    
    # print(f'coeff 1: {percentage:.2f}')
    print(f'{date}')
    coef_1_dict['date'] = date
    coef_1_dict['coeff'] = 1
    coef_1_dict['urel'] = urel_dict[run_type]
    coef_1_dict['tf'] = temp_dict[run_type]
    coef_1_dict['dt'] = dt
    coef_1_dict['Qib'] = (Qib_val/1e11)
    # coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg #(dQ_dt_dict[TF])
    coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg_dict[date]
    
    coef_1_dict['melt_rate_avg_m3s'] = Qib.melt_rate_integrated.data
    
    coef_1_dict['percentage'] = f'{(Qib_val/dQ_dt_HEL_CTD_avg_dict[date])*100:.2f}' #f'{(Qib_val/dQ_dt_dict[TF])*100:.2f}'
    
    series = pd.Series(coef_1_dict)
    series_list1.append(series)

df_50 = pd.DataFrame(series_list1, columns=cols)   



2016-04-24 00:00:00
2018-05-04 00:00:00
2020-09-03 00:00:00
2023-07-27 00:00:00
2024-08-22 00:00:00


In [52]:
!pwd

/home/m484s199/iceberg_py/data/iceberg_model_output_melt_fix/helheim/min


In [53]:
nc.split('_')

['2024-08-22', 'helheim', 'coeff', '1', 'CTD', 'constant', 'UREL', '07.nc']

In [54]:
df_50

,date,coeff,urel,tf,dt,Qib,Qaww,melt_rate_avg_m3s,percentage
0,2016-04-24,1,0.07,4.0,50,0.039369,2.247118e+11,11.752029126955307,1.75
1,2018-05-04,1,0.07,4.0,50,0.039734,2.423553e+11,11.860814459925852,1.64
2,2020-09-03,1,0.07,4.0,50,0.044188,2.396284e+11,13.190520775887242,1.84
3,2023-07-27,1,0.07,4.0,50,0.095832,2.423553e+11,28.606504746372366,3.95
4,2024-08-22,1,0.07,4.0,50,0.073407,2.273112e+11,21.91240464251306,3.23


In [55]:
df_50['percentage'].astype(np.float64).mean()

np.float64(2.482)

In [56]:
run_type = 'min'

dt = 150
psw = 1024 #kg m3
csw = 3974 #J kg-1 C-1
day2sec = 86400
depth = 450
temp = temp_dict[run_type]

def Qaw_calc(area, dt = dt, psw = psw, csw = csw, depth = depth, temp = temp):
    
    vol = area * depth
    
    Qaw = psw * csw * ( (vol * temp) / (dt * day2sec) )

    return Qaw

dQ_dt_HEL_CTD_avg_dict = {date:Qaw_calc(vol) for date, vol in area_dict.items()}

cols = ['coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'melt_rate_mday', 'percentage']

In [57]:
cols = ['date', 'coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'percentage']
coef_1_dict = {}

series_list1 = []
for i,nc in enumerate(coeff_1_list):
    
    Qib = xr.open_dataset(f'{coeff_1_path}{nc}')
    Qib_val = Qib.Qib.data
    date = nc[:10]
    date = pd.to_datetime(date)
    TF = nc.split('_')[6]
    urel_val = nc.split('_')[-1].split('.')[1]
    
    # print(f'coeff 1: {percentage:.2f}')
    print(f'{date}')
    coef_1_dict['date'] = date
    coef_1_dict['coeff'] = 1
    coef_1_dict['urel'] = urel_dict[run_type]
    coef_1_dict['tf'] = temp_dict[run_type]
    coef_1_dict['dt'] = dt
    coef_1_dict['Qib'] = (Qib_val/1e11)
    # coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg #(dQ_dt_dict[TF])
    coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg_dict[date]
    
    coef_1_dict['melt_rate_avg_m3s'] = Qib.melt_rate_integrated.data
    
    coef_1_dict['percentage'] = f'{(Qib_val/dQ_dt_HEL_CTD_avg_dict[date])*100:.2f}' #f'{(Qib_val/dQ_dt_dict[TF])*100:.2f}'
    
    series = pd.Series(coef_1_dict)
    series_list1.append(series)

df_150 = pd.DataFrame(series_list1, columns=cols)   



2016-04-24 00:00:00
2018-05-04 00:00:00
2020-09-03 00:00:00
2023-07-27 00:00:00
2024-08-22 00:00:00


In [58]:
df_150

,date,coeff,urel,tf,dt,Qib,Qaww,melt_rate_avg_m3s,percentage
0,2016-04-24,1,0.07,4.0,150,0.039369,7.490395e+10,11.752029126955307,5.26
1,2018-05-04,1,0.07,4.0,150,0.039734,8.078511e+10,11.860814459925852,4.92
2,2020-09-03,1,0.07,4.0,150,0.044188,7.987612e+10,13.190520775887242,5.53
3,2023-07-27,1,0.07,4.0,150,0.095832,8.078511e+10,28.606504746372366,11.86
4,2024-08-22,1,0.07,4.0,150,0.073407,7.577039e+10,21.91240464251306,9.69


In [59]:
df_150['percentage'].astype(np.float64).mean()

np.float64(7.452)